In [1]:
import pandas as pd
import numpy as np
import timeit
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import urllib.request
import zipfile
import os

Звантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет:

In [2]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
ZIP_FILE = "household_power_consumption.zip"
DATA_FILE = "household_power_consumption.txt"

def download_with_retry(url, filename, retries=5):
    for attempt in range(1, retries + 1):
        try:
            print(f"Спроба {attempt}...")
            urllib.request.urlretrieve(url, filename)
            print("Завантажено успішно.")
            return True
        except Exception as e:
            print(f"Помилка: {e}, пробую ще раз...")
            # Видаляю неповний файл щоб не було колізії
            if os.path.exists(filename):
                os.remove(filename)
    print("Не вдалось завантажити після всіх спроб.")
    return False

if not os.path.exists(DATA_FILE):
    if download_with_retry(URL, ZIP_FILE):
        with zipfile.ZipFile(ZIP_FILE, "r") as z:
            z.extractall(".")
        print("Розпаковано.")
else:
    print("Файл вже існує, пропускаю завантаження.")

Файл вже існує, пропускаю завантаження.


Здійснити data cleaning:

In [3]:
# Зчитую датасет та виконую data cleaning
df = pd.read_csv(
    DATA_FILE,
    sep=";",
    na_values=["?"],
    low_memory=False,
)

# Об'єдную Date та Time в один datetime стовпець
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")
df = df.drop(columns=["Date", "Time"])

# Переводжу всі числові колонки у float
num_cols = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]
df[num_cols] = df[num_cols].astype(float)

# Заповнюю пропуски медіаною кожного стовпця
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print(f"Розмір датасету: {df.shape}")
print(f"Пропуски після очищення:\n{df.isnull().sum()}")
df.head()

Розмір датасету: (2075259, 8)
Пропуски після очищення:
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
datetime                 0
dtype: int64


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт:

In [4]:
# Відбираю записи з активною потужністю понад 5 кВт
def query_high_power(df, threshold=5.0):
    result = df[df["Global_active_power"] > threshold]
    return result

time_q1 = timeit.timeit(lambda: query_high_power(df), number=10) / 10

result_q1 = query_high_power(df)
print(f"Кількість записів з Global_active_power > 5 кВт: {len(result_q1)}")
print(f"Середній час виконання: {time_q1:.4f} с")
result_q1.head()

Кількість записів з Global_active_power > 5 кВт: 17547
Середній час виконання: 0.0115 с


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00


Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер:

In [5]:
# Відбираю записи з силою струму 19-20 А, серед них ті, де Sub_metering_2 > Sub_metering_3
def query_current_and_appliances(df):
    # Фільтрую по силі струму
    mask_current = df["Global_intensity"].between(19, 20)
    subset = df[mask_current]

    # Фільтрую: пральна машина + холодильник > бойлер + кондиціонер
    mask_appliances = subset["Sub_metering_2"] > subset["Sub_metering_3"]
    result = subset[mask_appliances]
    return result

time_q2 = timeit.timeit(lambda: query_current_and_appliances(df), number=10) / 10

result_q2 = query_current_and_appliances(df)
print(f"Записів з струмом 19–20 А: {len(df[df['Global_intensity'].between(19, 20)])}")
print(f"З них Sub_metering_2 > Sub_metering_3: {len(result_q2)}")
print(f"Середній час виконання: {time_q2:.4f} с")
result_q2.head()

Записів з струмом 19–20 А: 7021
З них Sub_metering_2 > Sub_metering_3: 2509
Середній час виконання: 0.0201 с


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
45,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00


Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії:

In [6]:
# Обираю 500 000 випадкових записів без повторів, обчислюю середні по трьох групах споживання
def query_random_sample(df, n=500_000):
    sample = df.sample(n=n, replace=False, random_state=42)

    means = {
        "Sub_metering_1 (кухня)":                sample["Sub_metering_1"].mean(),
        "Sub_metering_2 (пральна+холодильник)":   sample["Sub_metering_2"].mean(),
        "Sub_metering_3 (бойлер+кондиціонер)":    sample["Sub_metering_3"].mean(),
    }
    return sample, means

time_q3 = timeit.timeit(lambda: query_random_sample(df), number=5) / 5

sample_q3, means_q3 = query_random_sample(df)
print(f"Розмір вибірки: {len(sample_q3)}")
print(f"Середній час виконання: {time_q3:.4f} с")
print("\nСередні значення по групах споживання:")
for name, val in means_q3.items():
    print(f"  {name}: {val:.4f} Вт·год")

Розмір вибірки: 500000
Середній час виконання: 0.3439 с

Середні значення по групах споживання:
  Sub_metering_1 (кухня): 1.1090 Вт·год
  Sub_metering_2 (пральна+холодильник): 1.2837 Вт·год
  Sub_metering_3 (бойлер+кондиціонер): 6.3971 Вт·год


Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини:

In [7]:
# Відбираю записи після 18:00 з споживанням понад 6 кВт, серед них де група 2 найбільша, потім кожен 3-й з першої половини та кожен 4-й з другої
def query_evening_heavy(df):
    # Фільтрую по часу — після 18:00
    mask_time = df["datetime"].dt.hour >= 18
    subset = df[mask_time]

    # Фільтрую по потужності — понад 6 кВт
    mask_power = subset["Global_active_power"] > 6
    subset = subset[mask_power]

    # Фільтрую — група 2 є найбільшою серед трьох
    mask_group2 = (
        (subset["Sub_metering_2"] > subset["Sub_metering_1"]) &
        (subset["Sub_metering_2"] > subset["Sub_metering_3"])
    )
    subset = subset[mask_group2].reset_index(drop=True)

    # Ділю на дві половини
    mid = len(subset) // 2
    first_half  = subset.iloc[:mid].iloc[::3]   # кожен 3-й
    second_half = subset.iloc[mid:].iloc[::4]   # кожен 4-й

    result = pd.concat([first_half, second_half]).reset_index(drop=True)
    return result

time_q4 = timeit.timeit(lambda: query_evening_heavy(df), number=5) / 5

result_q4 = query_evening_heavy(df)
print(f"Кількість відібраних записів: {len(result_q4)}")
print(f"Середній час виконання: {time_q4:.4f} с")
result_q4.head()

Кількість відібраних записів: 310
Середній час виконання: 0.2628 с


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
0,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
1,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
2,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
3,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
4,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00


Пронормувати та стандартизувати вибраний датасет:

In [8]:
# Нормалізую (MinMax) та стандартизую (Z-score) вибрані стовпці
def normalize_standardize(df, cols):
    result = df[cols].copy()

    # MinMax нормалізація — приводжу до діапазону [0, 1]
    scaler_mm = MinMaxScaler()
    normalized = pd.DataFrame(
        scaler_mm.fit_transform(result),
        columns=[f"{c}_norm" for c in cols]
    )

    # Z-score стандартизація — mean=0, std=1
    scaler_std = StandardScaler()
    standardized = pd.DataFrame(
        scaler_std.fit_transform(result),
        columns=[f"{c}_std" for c in cols]
    )

    return pd.concat([result, normalized, standardized], axis=1)

cols_to_scale = [
    "Global_active_power", "Global_intensity",
    "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]

result_scaled = normalize_standardize(df, cols_to_scale)
print("Нормалізовані та стандартизовані дані:")
result_scaled.head()

Нормалізовані та стандартизовані дані:


,Global_active_power,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Global_active_power_norm,Global_intensity_norm,Sub_metering_1_norm,Sub_metering_2_norm,Sub_metering_3_norm,Global_active_power_std,Global_intensity_std,Sub_metering_1_std,Sub_metering_2_std,Sub_metering_3_std
0,4.216,18.4,0.0,1.0,17.0,0.374796,0.377593,0.0,0.0125,0.548387,2.975591,3.120054,-0.181154,-0.048773,1.262163
1,5.360,23.0,0.0,1.0,16.0,0.478363,0.473029,0.0,0.0125,0.516129,4.062977,4.160250,-0.181154,-0.048773,1.143202
2,5.374,23.0,0.0,2.0,17.0,0.479631,0.473029,0.0,0.0250,0.548387,4.076284,4.160250,-0.181154,0.124020,1.262163
3,5.388,23.0,0.0,1.0,17.0,0.480898,0.473029,0.0,0.0125,0.548387,4.089591,4.160250,-0.181154,-0.048773,1.262163
4,3.666,15.8,0.0,1.0,17.0,0.325005,0.323651,0.0,0.0125,0.548387,2.452810,2.532116,-0.181154,-0.048773,1.262163


Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів:

In [9]:
# Обчислюю коефіцієнти кореляції Пірсона та Спірмена між Global_active_power та Global_intensity
def compute_correlations(df, col1, col2):
    pearson_r,  p_pearson  = stats.pearsonr(df[col1],  df[col2])
    spearman_r, p_spearman = stats.spearmanr(df[col1], df[col2])

    print(f"Кореляція між '{col1}' та '{col2}':")
    print(f"  Пірсон:  r = {pearson_r:.4f},  p = {p_pearson:.2e}")
    print(f"  Спірмен: r = {spearman_r:.4f}, p = {p_spearman:.2e}")
    return pearson_r, spearman_r

compute_correlations(df, "Global_active_power", "Global_intensity")

Кореляція між 'Global_active_power' та 'Global_intensity':
  Пірсон:  r = 0.9989,  p = 0.00e+00
  Спірмен: r = 0.9955, p = 0.00e+00


(np.float64(0.9988912417012629), np.float64(0.9955083176993444))

Провести One Hot Encoding категоріального атрибута:

In [10]:
# Створюю категоріальний атрибут часу доби та виконую One Hot Encoding
def one_hot_encoding(df):
    df_enc = df.copy()

    # Створюю категорію часу доби на основі години
    def time_category(hour):
        if 0  <= hour < 6:  return "ніч"
        if 6  <= hour < 12: return "ранок"
        if 12 <= hour < 18: return "день"
        return "вечір"

    df_enc["time_of_day"] = df_enc["datetime"].dt.hour.apply(time_category)

    print("Розподіл категорій:")
    print(df_enc["time_of_day"].value_counts())

    # Виконую One Hot Encoding
    ohe = pd.get_dummies(df_enc["time_of_day"], prefix="time")
    result = pd.concat([df_enc[["datetime", "Global_active_power"]], ohe], axis=1)
    return result

result_ohe = one_hot_encoding(df)
print("\nРезультат One Hot Encoding:")
result_ohe.head(10)

Розподіл категорій:
time_of_day
вечір    518943
день     518796
ніч      518760
ранок    518760
Name: count, dtype: int64

Результат One Hot Encoding:


,datetime,Global_active_power,time_вечір,time_день,time_ніч,time_ранок
0,2006-12-16 17:24:00,4.216,False,True,False,False
1,2006-12-16 17:25:00,5.360,False,True,False,False
2,2006-12-16 17:26:00,5.374,False,True,False,False
3,2006-12-16 17:27:00,5.388,False,True,False,False
4,2006-12-16 17:28:00,3.666,False,True,False,False
5,2006-12-16 17:29:00,3.520,False,True,False,False
6,2006-12-16 17:30:00,3.702,False,True,False,False
7,2006-12-16 17:31:00,3.700,False,True,False,False
8,2006-12-16 17:32:00,3.668,False,True,False,False
9,2006-12-16 17:33:00,3.662,False,True,False,False
